In [ ]:
%pip install -q tqdm

In [ ]:
from google.colab import drive
from pathlib import Path
import json, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from tqdm.auto import tqdm
from collections import defaultdict

drive.mount('/content/drive')

RAR_PATH       = Path('/content/drive/MyDrive/Dataset/CASIA-Webface/casia-webface.rar')
EXTRACT_DIR    = Path('/tmp/casia-webface')   # giải nén vào /tmp để I/O nhanh
CHECKPOINT_DIR = Path('/content/drive/MyDrive/ms1m_arcface_pipeline/checkpoints_casia')
LOG_DIR        = Path('/content/drive/MyDrive/ms1m_arcface_pipeline/logs_casia')
for p in [EXTRACT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

NUM_CLASSES    = 10_575
EMBEDDING_SIZE = 512
ARC_S, ARC_M, ARC_K        = 64.0, 0.5, 3
ARC_M_START, ARC_M_WARMUP  = 0.2, 5
BATCH_SIZE     = 128
EPOCHS         = 25
LR, MOMENTUM   = 0.05, 0.9
WEIGHT_DECAY   = 5e-4
LR_MILESTONES  = [12, 18, 22]
LR_GAMMA       = 0.1
MAX_GRAD_NORM  = 25.0
VAL_SPLIT      = 0.02     # 2% dùng làm val
NUM_WORKERS    = 2

BEST_PATH = CHECKPOINT_DIR / 'best_model.pth'
LAST_PATH = CHECKPOINT_DIR / 'last_model.pth'
LOG_PATH  = LOG_DIR / 'history.json'

Mounted at /content/drive


In [ ]:
import subprocess

# Kiểm tra đã giải nén chưa
already_extracted = any(EXTRACT_DIR.iterdir()) if EXTRACT_DIR.exists() else False

if already_extracted:
    print('Dataset đã được giải nén.')
else:
    print(f'Đang giải nén {RAR_PATH} ...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'unrar'], check=True)
    result = subprocess.run(
        ['unrar', 'x', '-y', str(RAR_PATH), str(EXTRACT_DIR) + '/'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Giải nén thất bại!')
    print('Giải nén xong.')

# Tìm thư mục gốc chứa các identity folder
# Cấu trúc: EXTRACT_DIR/[casia-webface/]<id>/<img>.jpg
DATA_ROOT = EXTRACT_DIR
subdirs = [d for d in EXTRACT_DIR.iterdir() if d.is_dir()]
if len(subdirs) == 1:
    DATA_ROOT = subdirs[0]   # vào thư mục con nếu có

n_ids = len([d for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'Số identity: {n_ids}')

Đang giải nén /content/drive/MyDrive/Dataset/CASIA-Webface/casia-webface.rar ...
Giải nén xong.
DATA_ROOT: /tmp/casia-webface/casia-webface
Số identity: 10572


In [ ]:
class ConvBN(nn.Module):
    def __init__(self, inp, oup, k=3, s=1, p=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(inp, oup, k, s, p, groups=groups, bias=False),
                                   nn.BatchNorm2d(oup), nn.PReLU(oup))
    def forward(self, x): return self.block(x)

class Bottleneck(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio):
        super().__init__()
        hidden = inp * expand_ratio
        self.use_res = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            nn.Conv2d(inp, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden),
            nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden),
            nn.Conv2d(hidden, oup, 1, bias=False), nn.BatchNorm2d(oup),
        )
    def forward(self, x): return x + self.conv(x) if self.use_res else self.conv(x)

class MobileFaceNet(nn.Module):
    CFG = [(2,64,5,2),(4,128,1,2),(2,128,6,1),(4,128,1,2),(2,128,2,1)]
    def __init__(self, emb=512):
        super().__init__()
        self.stem = nn.Sequential(ConvBN(3,64,s=2), ConvBN(64,64,groups=64))
        layers, inp = [], 64
        for t,c,n,s in self.CFG:
            for i in range(n):
                layers.append(Bottleneck(inp, c, s if i==0 else 1, t)); inp = c
        self.blocks    = nn.Sequential(*layers)
        self.conv_last = ConvBN(128, 512, k=1, s=1, p=0)
        self.gdc       = nn.Sequential(nn.Conv2d(512,512,7,groups=512,bias=False), nn.BatchNorm2d(512))
        self.output    = nn.Sequential(nn.Flatten(), nn.Linear(512,emb,bias=False), nn.BatchNorm1d(emb))
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d,nn.BatchNorm1d)): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.normal_(m.weight, 0, 0.01)
            elif isinstance(m, nn.PReLU): nn.init.constant_(m.weight, 0.25)
    def forward(self, x):
        return self.output(self.gdc(self.conv_last(self.blocks(self.stem(x)))))

class ArcFace(nn.Module):
    def __init__(self, emb, n_cls, s=64.0, m=0.5, k=3):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_cls * k, emb))
        nn.init.xavier_uniform_(self.W)
        self.s, self.m, self.k, self.n_cls = s, m, k, n_cls
    def forward(self, x, labels):
        x = F.normalize(x.float(), dim=1)
        W = F.normalize(self.W.float(), dim=1)
        cos, _ = F.linear(x, W).view(-1, self.n_cls, self.k).max(dim=2)
        cos = cos.clamp(-1+1e-4, 1-1e-4)
        oh  = torch.zeros_like(cos).scatter_(1, labels.view(-1,1), 1.0)
        out = cos*(1-oh) + torch.cos(torch.acos(cos)+self.m)*oh
        return F.cross_entropy(out*self.s, labels)
    def set_margin(self, m): self.m = m

In [ ]:
def get_margin(epoch):
    if epoch <= ARC_M_WARMUP:
        return ARC_M_START
    # Tăng tuyến tính từ epoch 5 → 10
    ramp_epochs = 5
    progress = min((epoch - ARC_M_WARMUP) / ramp_epochs, 1.0)
    return ARC_M_START + (ARC_M - ARC_M_START) * progress

In [ ]:
T_train = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.RandomGrayscale(0.05),
    transforms.ToTensor(),
    transforms.Normalize([.5,.5,.5],[.5,.5,.5]),
])
T_val = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([.5,.5,.5],[.5,.5,.5]),
])

full_dataset = datasets.ImageFolder(str(DATA_ROOT))  # transform=None (mặc định)
NUM_CLASSES  = len(full_dataset.classes)

n_val   = max(1, int(len(full_dataset) * VAL_SPLIT))
n_train = len(full_dataset) - n_val
train_subset, val_subset = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, i):
        img, label = self.subset[i]   # img là PIL vì full_dataset.transform=None
        return self.transform(img), label

train_ds = TransformSubset(train_subset, T_train)
val_ds   = TransformSubset(val_subset,   T_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers= True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers= True)

TRAIN_STEPS = len(train_loader)
print(f'NUM_CLASSES={NUM_CLASSES} | train={n_train} | val={n_val} | steps/epoch={TRAIN_STEPS}')

NUM_CLASSES=10572 | train=480811 | val=9812 | steps/epoch=3757


In [ ]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = torch.cuda.is_available()

model   = MobileFaceNet(EMBEDDING_SIZE).to(device)
arcface = ArcFace(EMBEDDING_SIZE, NUM_CLASSES, ARC_S, ARC_M_START, ARC_K).to(device)

optimizer = torch.optim.SGD(
    [{'params': model.parameters()}, {'params': arcface.parameters()}],
    lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)

LR_WARMUP_EPOCHS = 3

def get_lr(epoch, step_in_epoch):
    if epoch <= LR_WARMUP_EPOCHS:
        total = LR_WARMUP_EPOCHS * TRAIN_STEPS
        current = (epoch - 1) * TRAIN_STEPS + step_in_epoch + 1
        return LR * current / total
    factor = 1.0
    for ms in LR_MILESTONES:
        if epoch > ms: factor *= LR_GAMMA
    return LR * factor

def set_lr(lr):
    for g in optimizer.param_groups:
        g['lr'] = lr

scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
history, best_cos_gap, start_epoch, global_step = [], -float('inf'), 1, 0

if LAST_PATH.exists():
    ckpt = torch.load(str(LAST_PATH), map_location=device, weights_only=False)
    if all(torch.isfinite(v).all() for v in ckpt['model_state_dict'].values() if isinstance(v, torch.Tensor)):
        model.load_state_dict(ckpt['model_state_dict'])
        arcface.load_state_dict(ckpt['arcface_state_dict'])
        optimizer.load_state_dict(ckpt.get('optimizer_state_dict', {}))
        best_cos_gap = float(ckpt.get('best_cos_gap', best_cos_gap))
        start_epoch  = int(ckpt.get('epoch', 0)) + 1
        global_step  = int(ckpt.get('global_step', 0))
        if LOG_PATH.exists(): history = json.loads(LOG_PATH.read_text())
        print(f'Resumed epoch {start_epoch}, step {global_step}')

print(f'Device: {device} | {EPOCHS} epochs | {NUM_CLASSES} classes')

Resumed epoch 10, step 33813
Device: cuda | 25 epochs | 10572 classes


In [ ]:
@torch.no_grad()
def val_cos_gap(model):
    model.eval()
    id_embs = defaultdict(list)
    for imgs, lbls in val_loader:
        for e, l in zip(F.normalize(model(imgs.to(device)).float(), dim=1).cpu(), lbls):
            if len(id_embs[l.item()]) < 8: id_embs[l.item()].append(e)
    valid = {k:v for k,v in id_embs.items() if len(v)>=4}
    if len(valid) < 2: return None
    ids = list(valid.keys())
    intra = [(torch.stack(v)[i]*torch.stack(v)[j]).sum().item()
             for k,v in valid.items() for i in range(len(v)) for j in range(i+1,len(v))][:1000]
    inter = [((random.choice(valid[a])*random.choice(valid[b])).sum().item())
             for a,b in [random.sample(ids,2) for _ in range(1000)]]
    gap = sum(intra)/len(intra) - sum(inter)/len(inter)
    print(f'  cos_gap={gap:.4f}')
    return gap

for epoch in range(start_epoch, EPOCHS+1):
    arcface.set_margin(get_margin(epoch))
    model.train(); arcface.train()
    loss_sum = gn_sum = steps = 0

    for step, (imgs, lbls) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')):
        set_lr(get_lr(epoch, step))
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            emb  = model(imgs)
            loss = arcface(emb, lbls)
        if not torch.isfinite(loss): continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            list(model.parameters())+list(arcface.parameters()), MAX_GRAD_NORM)
        if torch.isfinite(grad_norm):
            scaler.step(optimizer)
            gn_sum += float(grad_norm)
        scaler.update()
        global_step += 1
        loss_sum += loss.item(); steps += 1

    # Val loss
    model.eval(); arcface.eval()
    vl_sum = vl_steps = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            loss = arcface(model(imgs.to(device)), lbls.to(device))
            if torch.isfinite(loss): vl_sum += loss.item(); vl_steps += 1

    tr_loss = loss_sum/max(steps,1)
    vl_loss = vl_sum/max(vl_steps,1)
    lr      = optimizer.param_groups[0]['lr']
    gap     = val_cos_gap(model)  # luôn tính cos_gap mỗi epoch

    state = {'epoch':epoch,'global_step':global_step,'best_cos_gap':best_cos_gap,
             'model_state_dict':model.state_dict(),'arcface_state_dict':arcface.state_dict(),
             'optimizer_state_dict':optimizer.state_dict()}
    torch.save(state, LAST_PATH)
    if gap is not None and gap > best_cos_gap:
        best_cos_gap = gap;
        state['best_cos_gap'] = gap
        torch.save(state, BEST_PATH); print(f'  best cos_gap={gap:.4f}')

    history.append({'epoch':epoch,
                    'tr_loss':round(tr_loss,4),
                    'vl_loss':round(vl_loss,4),
                    'lr':lr,'cos_gap':round(gap,4) if gap else None})
    LOG_PATH.write_text(json.dumps(history, indent=2))
    print(f'Ep {epoch:3d} | loss={tr_loss:.4f} | val={vl_loss:.4f} | lr={lr:.2e} | gnorm={gn_sum/max(steps,1):.2f}')

print('Done. Best model:', BEST_PATH)


Epoch 10/25:   0%|          | 0/3757 [00:00<?, ?it/s]

  cos_gap=0.4240
Ep  10 | loss=22.1700 | val=21.6101 | lr=5.00e-02 | gnorm=222.78


Epoch 11/25:   0%|          | 0/3757 [00:00<?, ?it/s]

  cos_gap=0.4326
  best cos_gap=0.4326
Ep  11 | loss=19.6140 | val=21.5988 | lr=5.00e-02 | gnorm=248.16


Epoch 12/25:   0%|          | 0/3757 [00:00<?, ?it/s]

  cos_gap=0.4131
Ep  12 | loss=19.1956 | val=22.2337 | lr=5.00e-02 | gnorm=116.84


Epoch 13/25:   0%|          | 0/3757 [00:00<?, ?it/s]

  cos_gap=0.4871
  best cos_gap=0.4871
Ep  13 | loss=13.8262 | val=14.9278 | lr=5.00e-03 | gnorm=8.30


Epoch 14/25:   0%|          | 0/3757 [00:00<?, ?it/s]